# CLM-0.4-mini M1-v2 Calibration

**Two-stage development protocol.** V1 seed `90401` is historical diagnosis only. V2 uses fresh development seed `90402`. Formal seeds `90411/90412/90413` must never be opened here.

This notebook first reconstructs seed-independent v2 data and checks the committed asset lock. If the repository still says `DATA_LOCK_PENDING`, stop after Phase A and commit the printed hashes before running Phase B.

After calibration, the final cell curates and pushes analysis-sized results to a dedicated GitHub branch. Checkpoints and raw 30M-token shards are excluded from Git.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path("/kaggle/working/mini-cells")
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/ArcheLabs/mini-cells.git", str(ROOT)], check=True)
else:
    subprocess.run(["git", "switch", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=ROOT, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT) + "[lm]"], check=True)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA = Path("/kaggle/working/clm-0.4-mini-m1-v2-data")
OUT = ROOT / "results" / "clm-0.4-mini-m1-v2-calibration"
VALIDATION = ROOT / "research" / "validations" / "clm-0.4-mini-m1-v2-language-validation"
REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"

print("ROOT:", ROOT)
print("DATA:", DATA)
print("OUT :", OUT)


In [ ]:
import torch
print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
})
assert torch.cuda.is_available(), "M1-v2 development calibration requires CUDA"


## Phase A — seed-independent v2 data

In [ ]:
if not (DATA / "asset-summary.json").is_file():
    subprocess.run([
        sys.executable,
        str(ROOT / "scripts/research/prepare_clm_0_4_mini_v2_data.py"),
        "--dataset-revision", REVISION,
        "--out", str(DATA),
    ], cwd=ROOT, check=True)

assets = json.loads((DATA / "asset-summary.json").read_text())
print(json.dumps(assets, indent=2, sort_keys=True))

lock = json.loads((VALIDATION / "asset-lock.json").read_text())
print("\nCommitted asset lock status:", lock["lock_status"])
if lock["lock_status"] != "LOCKED":
    print("\nSTOP HERE. Send/commit the asset-summary hashes before opening seed 90402.")
else:
    print("Asset lock is committed; Phase B may proceed.")


In [ ]:
# Safe plan-only validation. This never opens 90402.
PLAN_OUT = Path("/kaggle/working/clm-0.4-mini-m1-v2-plan")
subprocess.run([
    sys.executable,
    str(ROOT / "scripts/research/run.py"),
    "clm-0.4-mini-m1-v2-calibration",
    "--plan-only",
    "--out", str(PLAN_OUT),
], cwd=ROOT, check=True)
print((PLAN_OUT / "decision.json").read_text())


## Phase B — open development seed 90402

Run this cell **only after** `asset-lock.json` is `LOCKED` on the current repository commit. The runner refuses to proceed otherwise.


In [ ]:
lock = json.loads((VALIDATION / "asset-lock.json").read_text())
assert lock["lock_status"] == "LOCKED", "DATA_LOCK_PENDING: do not open 90402 yet."

subprocess.run([
    sys.executable,
    str(ROOT / "scripts/research/run.py"),
    "clm-0.4-mini-m1-v2-calibration",
    "--data-dir", str(DATA),
    "--out", str(OUT),
    "--device", "cuda",
    "--devices", ",".join(f"cuda:{i}" for i in range(torch.cuda.device_count())),
    "--seed", "90402",
    "--confirm-development-seed", "90402",
], cwd=ROOT, check=True)


In [ ]:
subprocess.run([
    sys.executable,
    str(ROOT / "scripts/research/report.py"),
    "clm-0.4-mini-m1-v2-calibration",
    "--results", str(OUT),
], cwd=ROOT, check=True)

decision = json.loads((OUT / "decision.json").read_text())
print(json.dumps(decision, indent=2, sort_keys=True))
print("\n", OUT / "RESULTS.md")


## Publish curated result branch

This restores the repository-result workflow. It pushes JSON/JSONL/CSV/MD/PNG evidence only; `.pt` checkpoints and raw data are excluded.

The Kaggle secret should contain a GitHub token with **Contents: Read and write**. The existing default secret name is `GITHUB_TOKEN`.


In [ ]:
PUSH_RESULTS = True

if PUSH_RESULTS:
    subprocess.run([
        sys.executable,
        str(ROOT / "scripts/research/publish.py"),
        "clm-0.4-mini-m1-v2-calibration",
        "--results", str(OUT),
        "--push",
    ], cwd=ROOT, check=True)
else:
    print("Set PUSH_RESULTS=True to publish the curated result branch.")
